# Model A: nhận diện yêu cầu và phân loại nhu cầu

**Chỉ cần notebook này và `data_model_a.xlsx`.** Toàn bộ code nằm trong các ô bên dưới.
Không chạy file 00, không upload ZIP code, không cần notebook xuất model riêng.

Đầu vào: câu tiếng Việt. Đầu ra: `is_request`, điểm yêu cầu và nhiều nhãn `RESCUE`, `MEDICAL`, `EVACUATION`, `SUPPLY`. Kiến trúc: PhoBERT + hai đầu sigmoid; fine-tune cả encoder và hai đầu.

Backbone: `vinai/phobert-base-v2`. PyVi tách từ tiếng Việt; tokenizer PhoBERT dạng slow.
Mỗi model có file Excel riêng. Một phần câu nguồn và tập đánh giá của hai bộ hiện tại trùng nhau;
A có 960 mẫu train, B có 1.032 mẫu train, mỗi bộ có 176 validation và 160 test.

**Mẫu ban đầu là dữ liệu tổng hợp, chưa được người duyệt độc lập.** Điểm test không chứng minh chất lượng với tin SOS thực tế.
Baseline lần đầu là encoder đã tiền huấn luyện + đầu tác vụ khởi tạo ngẫu nhiên;
PhoBERT nguyên bản chưa có khả năng xuất các nhãn SOS này. Đây là đối chiếu trước/sau, không phải so với một model SOS đã huấn luyện.

Chọn **Runtime → Change runtime type → GPU**, rồi chạy các ô theo thứ tự.
Lần đầu dùng `new`. Khi thêm dữ liệu, xem bước 12, dùng lại chính notebook này ở chế độ `continue`.

## 1. Cài thư viện

Giữ PyTorch/GPU có sẵn của Colab. Tokenizers được cài từ wheel, tránh lỗi build Rust.
Nếu ô báo cần restart: chọn **Runtime → Restart session**, rồi chạy lại từ ô này một lần.
Không dùng yêu cầu `transformers==4.44.2` của notebook cũ.

In [ ]:
import os, sys, subprocess, importlib.metadata
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
wanted = {'transformers': '4.57.6', 'tokenizers': '0.22.2'}
stale = [name for name, version in wanted.items()
         if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--only-binary=tokenizers',
    'transformers==4.57.6', 'tokenizers==0.22.2', 'pyvi==0.1.1', 'openpyxl==3.1.5',
    'safetensors>=0.4.5', 'scikit-learn>=1.5,<2', 'pandas', 'matplotlib', 'tqdm'], check=True)
if stale:
    raise RuntimeError('Đã cài xong. Chọn Runtime → Restart session rồi chạy lại từ ô 1. Đang nạp bản cũ: ' + ', '.join(stale))
import torch, transformers, tokenizers
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
print('Transformers:', transformers.__version__, '| Tokenizers:', tokenizers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Chưa bật GPU')
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime → Change runtime type → GPU rồi chạy lại. Chưa bắt đầu train.')


## 2. Kết nối Drive và cấu hình

Lần đầu giữ `TRAIN_MODE=new`, `EPOCHS=5`, `LEARNING_RATE=0.00002`.
Học tiếp: `TRAIN_MODE=continue`, điền `PARENT_MODEL`, gợi ý `EPOCHS=3`, `LEARNING_RATE=0.000005`.
Các ô số chỉ nhận số, không nhập biểu thức `if ... else ...`. Notebook giữ đúng giá trị bạn điền.

`RUN_NAME` trống sẽ tự tạo tên mới. `PARENT_MODEL` là đường dẫn tới thư mục **model_a** có `metadata.json`, không phải ZIP.
`UPLOAD_EXCEL=True` để chọn file từ máy. Lần sau đã lưu trên Drive thì đặt False; muốn dùng Excel vừa sửa thì bật lại True.
Nếu Drive báo `mount failed`, thử `REMOUNT_DRIVE=True` và chạy lại ô này, hoàn tất hộp thoại cấp quyền.
Nếu vẫn thất bại, lưu notebook rồi chọn Runtime → Disconnect and delete runtime, kết nối lại GPU và chạy từ bước 1.
Thao tác xóa runtime làm mất file tạm trong /content; file đã lưu trên Google Drive vẫn được giữ.

In [ ]:
#@title Cấu hình model A
from pathlib import Path
from datetime import datetime
from google.colab import drive, files
REMOUNT_DRIVE = False #@param {type:"boolean"}
try:
    drive.mount('/content/drive', force_remount=REMOUNT_DRIVE, timeout_ms=180000)
except (ValueError, OSError) as exc:
    raise RuntimeError(
        'Chưa kết nối được Google Drive; chưa bắt đầu train. '
        'Thử REMOUNT_DRIVE=True và chạy lại ô này, hoàn tất cấp quyền Google. '
        'Nếu vẫn lỗi: lưu notebook, chọn Runtime → Disconnect and delete runtime, '
        'kết nối GPU rồi chạy từ bước 1. File tạm /content sẽ mất, file đã lưu trên Drive vẫn giữ. '
        'Nếu còn lỗi, gửi toàn bộ thông báo phía trên mount failed. '
        'Chi tiết gốc: ' + str(exc)
    ) from exc
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive chưa có MyDrive. Dừng tại đây, không tạo thư mục lưu model trên ổ tạm.')
WORK = Path('/content/drive/MyDrive/SOS_Thesis')
WORK.mkdir(parents=True, exist_ok=True)
TASK = 'a'
TRAIN_MODE = 'new' #@param ["new", "continue"]
EPOCHS = 5 #@param {type:"integer"}
BATCH_SIZE = 8 #@param {type:"integer"}
LEARNING_RATE = 0.00002 #@param {type:"number"}
SEED = 42 #@param {type:"integer"}
RUN_NAME = '' #@param {type:"string"}
PARENT_MODEL = '' #@param {type:"string"}
UPLOAD_EXCEL = True #@param {type:"boolean"}
EVALUATION_DOMAIN = 'synthetic' #@param ["synthetic", "real_vietnamese"]
EXCEL = WORK / 'data_model_a.xlsx'
resolved_name = RUN_NAME.strip() or ('model_a_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
if Path(resolved_name).name != resolved_name or resolved_name in {'.', '..'}:
    raise ValueError('RUN_NAME chỉ là tên, không chứa đường dẫn.')
RUN = WORK / 'runs' / resolved_name
if TRAIN_MODE not in {'new', 'continue'}:
    raise ValueError('TRAIN_MODE chỉ là new hoặc continue')
if TRAIN_MODE == 'continue' and not PARENT_MODEL.strip():
    raise ValueError('Điền PARENT_MODEL khi học tiếp.')
PARENT = Path(PARENT_MODEL.strip()) if TRAIN_MODE == 'continue' else None
if PARENT and not PARENT.is_absolute():
    PARENT = WORK / PARENT
if TRAIN_MODE == 'new' and PARENT_MODEL.strip():
    raise ValueError('Bạn đã điền model cha. Chuyển TRAIN_MODE=continue hoặc xóa PARENT_MODEL.')
print('Dữ liệu:', EXCEL, '\nRun mới:', RUN, '\nModel cha:', PARENT)


## 3. Nạp file Excel của model A

Chọn **`data_model_a.xlsx`**. File được lưu trên Drive để dùng lần sau.
Nếu cập nhật Excel, bản cũ được giữ trong thư mục `data_history` trên Drive; bản dữ liệu của mỗi lần train cũng nằm trong run đó.

In [ ]:
import shutil
if UPLOAD_EXCEL:
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.xlsx'):
        raise ValueError('Chọn đúng một file .xlsx của model này.')
    payload = next(iter(uploaded.values()))
    if EXCEL.exists() and EXCEL.read_bytes() != payload:
        backup = WORK / 'data_history' / (EXCEL.stem + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '.xlsx')
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(EXCEL, backup)
    EXCEL.write_bytes(payload)
if not EXCEL.is_file():
    raise FileNotFoundError('Chưa có Excel trên Drive. Bật UPLOAD_EXCEL=True ở bước 2 và chạy lại bước 3.')
print('Đã nạp:', EXCEL)


## 4. Đọc và kiểm tra nhãn

Ô dưới định nghĩa cách đọc Excel và chống rò rỉ dữ liệu. Bạn chỉ cần chạy, không cần chỉnh.
Có lỗi sẽ báo tên sheet, dòng và nguyên nhân trước khi tải model/train.

In [ ]:
"""Raw-text span schema and leakage checks. No ML dependencies required."""
import hashlib
import json
import re
import unicodedata
from pathlib import Path

NEEDS = ['RESCUE', 'MEDICAL', 'EVACUATION', 'SUPPLY']
ENTITIES = ['LOCATION', 'PEOPLE_COUNT', 'VULNERABLE_GROUP', 'RESOURCE']
BIO = ['O'] + [f'{prefix}-{kind}' for kind in ENTITIES for prefix in ['B', 'I']]


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8-sig').splitlines() if line.strip()]


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')


def fingerprint(text):
    value = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', text).casefold()).strip()
    return hashlib.sha256(value.encode()).hexdigest()


In [ ]:
"""Task-specific Excel inputs. This source is embedded in both standalone notebooks."""
import hashlib
import json
import re
import unicodedata
from pathlib import Path
from openpyxl import load_workbook


def parse_marked_text(value):
    """Convert [LOCATION|xã A] into raw text and Unicode spans; no manual offsets."""
    value = unicodedata.normalize('NFC', str(value))
    pattern = re.compile(r'\[([A-Z_]+)\|([^\[\]]+)\]')
    chunks, entities, cursor, length = [], [], 0, 0
    for match in pattern.finditer(value):
        prefix = value[cursor:match.start()]
        if '[' in prefix or ']' in prefix:
            raise ValueError('Dấu nhãn không hợp lệ. Dùng [LOCATION|cụm từ], không lồng nhãn.')
        chunks.append(prefix)
        length += len(prefix)
        label, text = match.groups()
        if label not in ENTITIES or not text.strip() or text != text.strip():
            raise ValueError(f'Nhãn/cụm từ không hợp lệ: {match.group()}')
        entities.append({'start': length, 'end': length + len(text), 'label': label, 'text': text})
        chunks.append(text)
        length += len(text)
        cursor = match.end()
    tail = value[cursor:]
    if '[' in tail or ']' in tail:
        raise ValueError('Dấu nhãn chưa đóng hoặc sai cú pháp [NHAN|cụm từ].')
    chunks.append(tail)
    return ''.join(chunks), entities


def excel_bool(value):
    if value is True or str(value).strip().lower() in {'1', 'true', 'yes', 'có', 'co'}:
        return True
    if value is False or str(value).strip().lower() in {'0', 'false', 'no', 'không', 'khong'}:
        return False
    raise ValueError(f'Giá trị phải là 0 hoặc 1: {value!r}')


def load_excel_data(path, task, evaluation_domain='synthetic'):
    if task not in {'a', 'b'}:
        raise ValueError('task phải là a hoặc b')
    if evaluation_domain not in {'synthetic', 'real_vietnamese'}:
        raise ValueError('evaluation_domain không hợp lệ')
    workbook = load_workbook(path, read_only=True, data_only=False)
    required = ['id', 'group_id', 'text' if task == 'a' else 'text_annotated']
    required += ['is_request', 'needs'] if task == 'a' else []
    required += ['origin', 'reviewed', 'source']
    splits = {}
    seen_ids, seen_text, seen_groups = set(), {}, {}
    try:
        for sheet_name in ['Train', 'Validation', 'Test']:
            if sheet_name not in workbook.sheetnames:
                raise ValueError(f'Thiếu sheet {sheet_name}')
            iterator = workbook[sheet_name].iter_rows(values_only=True)
            headers = next(iterator, ())
            if len(set(headers)) != len(headers) or any(h not in headers for h in required):
                raise ValueError(f'{sheet_name}: cột bắt buộc: {required}. Không dùng file của model còn lại.')
            rows = []
            for number, values in enumerate(iterator, 2):
                if all(v is None or v == '' for v in values):
                    continue
                cells = dict(zip(headers, values))
                try:
                    if any(isinstance(v, str) and v.startswith('=') for v in values):
                        raise ValueError('Không dùng công thức Excel trong dữ liệu train.')
                    for column in required:
                        if column != 'needs' and (cells.get(column) is None or str(cells[column]).strip() == ''):
                            raise ValueError(f'Thiếu {column}')
                    if task == 'a':
                        text = unicodedata.normalize('NFC', str(cells['text']))
                        entities = []
                        request = excel_bool(cells['is_request'])
                        needs = [v.strip() for v in str(cells.get('needs') or '').split(';') if v.strip()]
                        if set(needs) - set(NEEDS) or len(needs) != len(set(needs)):
                            raise ValueError('needs chỉ dùng RESCUE;MEDICAL;EVACUATION;SUPPLY, không lặp nhãn.')
                        if not request and needs:
                            raise ValueError('is_request=0 phải để needs trống.')
                    else:
                        text, entities = parse_marked_text(cells['text_annotated'])
                        request, needs = bool(entities), []  # Internal collator placeholder; B does not learn request/needs.
                    if not text.strip():
                        raise ValueError('Nội dung trống')
                    origin = str(cells['origin']).strip()
                    reviewed = excel_bool(cells['reviewed'])
                    if origin not in {'synthetic', 'real', 'translated'}:
                        raise ValueError('origin phải là synthetic, real hoặc translated')
                    if origin != 'synthetic' and not reviewed:
                        raise ValueError('Dữ liệu thực/dịch cần được người gắn nhãn kiểm tra, rồi đặt reviewed=1.')
                    if sheet_name != 'Train':
                        expected = 'synthetic' if evaluation_domain == 'synthetic' else 'real'
                        if origin != expected:
                            raise ValueError(f'Chế độ đánh giá này yêu cầu origin={expected} cho Validation/Test.')
                    row = {'id': str(cells['id']).strip(), 'group_id': str(cells['group_id']).strip(),
                           'text': text, 'is_request': request, 'needs': needs, 'entities': entities,
                           'origin': origin, 'reviewed': reviewed, 'source_url': str(cells['source']).strip()}
                    key = fingerprint(text)
                    if row['id'] in seen_ids or key in seen_text:
                        raise ValueError(f'Trùng ID/nội dung (bản trước: {seen_text.get(key, row["id"])}).')
                    group = row['group_id']
                    if group in seen_groups and seen_groups[group] != sheet_name:
                        raise ValueError(f'group_id={group} xuất hiện ở nhiều tập: rò rỉ dữ liệu.')
                    seen_ids.add(row['id'])
                    seen_text[key] = row['id']
                    seen_groups[group] = sheet_name
                    rows.append(row)
                except (ValueError, TypeError) as exc:
                    raise ValueError(f'{sheet_name}, dòng Excel {number}: {exc}') from exc
            if not rows:
                raise ValueError(f'{sheet_name}: không có dữ liệu')
            if task == 'a':
                if {r['is_request'] for r in rows} != {True, False}:
                    raise ValueError(f'{sheet_name}: cần cả is_request=0 và 1')
                for need in NEEDS:
                    flags = [need in r['needs'] for r in rows]
                    if not any(flags) or all(flags):
                        raise ValueError(f'{sheet_name}: cần cả mẫu có và không có nhãn {need}')
            else:
                present = {e['label'] for r in rows for e in r['entities']}
                if set(ENTITIES) - present or all(r['entities'] for r in rows):
                    raise ValueError(f'{sheet_name}: cần đủ 4 loại thông tin và mẫu không có thông tin cần trích xuất')
            splits[sheet_name.lower()] = rows
    finally:
        workbook.close()
    return splits


def semantic_hash(rows, task):
    """Sorting Excel rows or changing its formatting does not change this fingerprint."""
    fields = ['id', 'group_id', 'text'] + (['is_request', 'needs'] if task == 'a' else ['entities'])
    records = [{k: row[k] for k in fields} for row in sorted(rows, key=lambda r: r['id'])]
    return hashlib.sha256(json.dumps(records, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()


def check_excel_parent(parent, splits, task):
    old = parent.get('excel_split_sha256', {})
    for name in ['validation', 'test']:
        if old.get(name) != semantic_hash(splits[name], task):
            raise ValueError(f'{name}: không khớp dữ liệu đánh giá của model cha. Dùng lại Validation/Test của lần trước. '
                             'Học tiếp trong bộ Excel này yêu cầu model xuất từ notebook mới.')


In [ ]:
import pandas as pd
from IPython.display import display
SPLITS = load_excel_data(EXCEL, TASK, EVALUATION_DOMAIN)
display(pd.DataFrame([{'Tập': k, 'Số mẫu': len(v)} for k, v in SPLITS.items()]))
display(pd.DataFrame(SPLITS['train'][:5])[['id', 'text', 'is_request', 'needs'] if TASK == 'a' else ['id', 'text', 'entities']])
print('Đánh giá trên:', EVALUATION_DOMAIN)


## 5. Tiền xử lý và định nghĩa mô hình

Các ô code bên dưới chứa trực tiếp cách tách từ, gắn BIO, mô hình và tính F1.
Dùng cùng cách tiền xử lý với API của hệ thống. Câu vượt 256 BPE hoặc nhãn cắt giữa từ sẽ được báo để sửa, không tự cắt mất nhãn.
Ngưỡng A được chọn trên Validation. F1 của B yêu cầu đúng cả loại nhãn và vị trí cụm từ.

In [ ]:
"""One preprocessing implementation for training and serving, including raw offsets.

PhoBERT has a slow tokenizer: do NOT call word_ids()/offset_mapping on it.
We tokenize each segmented word and supervise its first BPE piece only.
"""
import re
import unicodedata

MAX_LENGTH = 256
PREPROCESSOR = 'pyvi-0.1.1-nfc-offset-v1'


def segmented_words(text, segmenter=None):
    if text != unicodedata.normalize('NFC', text):
        raise ValueError('Text must be NFC normalized.')
    if '_' in text:
        raise ValueError('Literal underscores are unsupported in v1. Review/normalize the source first.')
    if segmenter is None:
        from pyvi import ViTokenizer
        segmenter = ViTokenizer.tokenize
    tokens = segmenter(text).split()
    result, cursor = [], 0
    for token in tokens:
        parts = token.split('_')
        pattern = r'\s+'.join(re.escape(p) for p in parts)
        while cursor < len(text) and text[cursor].isspace():
            cursor += 1
        match = re.compile(pattern).match(text, cursor)
        if not match:
            raise ValueError(f'Segmenter changed source characters near offset {cursor}; review the text.')
        result.append((token, match.start(), match.end()))
        cursor = match.end()
    if text[cursor:].strip():
        raise ValueError('Segmenter dropped source text.')
    return result


def encode(text, tokenizer, entities=None, segmenter=None):
    words = segmented_words(text, segmenter)
    labels = ['O'] * len(words)
    if entities is not None:
        for ent in entities:
            indices = [i for i, (_, a, b) in enumerate(words) if a < ent['end'] and b > ent['start']]
            if not indices or words[indices[0]][1] != ent['start'] or words[indices[-1]][2] != ent['end']:
                raise ValueError(f'Entity boundary cuts a segmented word: {ent}. Review annotation.')
            for j, i in enumerate(indices):
                labels[i] = ('B-' if j == 0 else 'I-') + ent['label']
    ids, targets, first = [tokenizer.bos_token_id], [-100], []
    for i, (word, _, _) in enumerate(words):
        pieces = tokenizer.encode(word, add_special_tokens=False)
        if not pieces:
            raise ValueError('Empty BPE word')
        first.append(len(ids))
        ids.extend(pieces)
        targets.extend([BIO.index(labels[i])] + [-100] * (len(pieces) - 1))
    ids.append(tokenizer.eos_token_id)
    targets.append(-100)
    if len(ids) > MAX_LENGTH:
        raise ValueError(f'Text has {len(ids)} BPE tokens; v1 accepts at most {MAX_LENGTH}. Split and relabel first.')
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': targets,
            'words': words, 'first': first}


def decode_spans(text, words, tags):
    spans, current = [], None
    for (_, a, b), tag in zip(words, tags):
        if tag == 'O':
            current = None
            continue
        prefix, kind = tag.split('-', 1)
        if prefix == 'I' and current is not None and current['label'] == kind:
            current['end'] = b
            current['text'] = text[current['start']:b]
        else:
            current = {'start': a, 'end': b, 'label': kind, 'text': text[a:b]}
            spans.append(current)
    return spans


In [ ]:
"""Model A: one encoder + two classification heads. Model B uses HF token classifier."""
import json
from pathlib import Path
import torch
from torch import nn
from transformers import AutoModel
from safetensors.torch import save_file, load_file


class RequestNeedsModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({'request': nn.Linear(encoder.config.hidden_size, 1),
                                    'needs': nn.Linear(encoder.config.hidden_size, len(NEEDS))})

    def forward(self, input_ids, attention_mask):
        pooled = self.dropout(self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0])
        return self.heads['request'](pooled).squeeze(-1), self.heads['needs'](pooled)

    def save_bundle(self, directory, tokenizer, metadata):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        self.encoder.save_pretrained(directory / 'encoder', safe_serialization=True)
        tokenizer.save_pretrained(directory)
        save_file({k: v.detach().cpu().contiguous() for k, v in self.heads.state_dict().items()}, str(directory / 'heads.safetensors'))
        (directory / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

    @classmethod
    def from_bundle(cls, directory):
        directory = Path(directory)
        model = cls(AutoModel.from_pretrained(directory / 'encoder', local_files_only=True))
        model.heads.load_state_dict(load_file(str(directory / 'heads.safetensors')))
        return model


In [ ]:
"""Warm-start from a complete model artifact; not an optimizer-state resume."""
import hashlib
import json
from pathlib import Path


def read_parent(directory, task, demo=False):
    directory = Path(directory)
    raw = (directory / 'metadata.json').read_bytes()
    meta = json.loads(raw)
    if meta.get('task') != task:
        raise ValueError(f'Checkpoint is not model {task.upper()}')
    if meta.get('schema_version') != 1 or meta.get('preprocessor') != PREPROCESSOR or meta.get('max_length') != MAX_LENGTH:
        raise ValueError('Checkpoint preprocessing/schema differs; cannot continue with these labels.')
    expected_key, expected_labels = ('needs', NEEDS) if task == 'a' else ('bio_labels', BIO)
    if meta.get(expected_key) != expected_labels:
        raise ValueError('Checkpoint label names/order differ.')
    if meta.get('demo') and not demo:
        raise ValueError('Tiny demo checkpoint cannot initialize a non-demo run.')
    return meta, {'path': str(directory.resolve()), 'run_id': meta['run_id'],
                  'metadata_sha256': hashlib.sha256(raw).hexdigest()}


def load_parent_model(directory, task):
    from transformers import AutoModelForTokenClassification
    if task == 'a':
        return RequestNeedsModel.from_bundle(directory)
    model = AutoModelForTokenClassification.from_pretrained(directory, local_files_only=True)
    if [model.config.id2label.get(i) for i in range(len(BIO))] != BIO:
        raise ValueError('Model B checkpoint config has incompatible label order.')
    return model


In [ ]:
"""Evaluation and batching shared by both standalone notebooks."""
import numpy as np
import torch
from sklearn.metrics import f1_score, classification_report


def collate(items, pad_id):
    size = max(len(item['input_ids']) for item in items)
    result = {}
    for key, padding in [('input_ids', pad_id), ('attention_mask', 0), ('labels', -100)]:
        result[key] = torch.tensor([item[key] + [padding] * (size - len(item[key])) for item in items])
    result['targets'] = torch.tensor([[float(item['row']['is_request'])] +
        [float(n in item['row']['needs']) for n in NEEDS] for item in items])
    result['items'] = items
    return result


def choose_thresholds(y, p):
    # Thresholds are chosen ONLY from validation. Scores are not calibrated probabilities.
    values = []
    for j in range(5):
        candidates = np.arange(0.2, 0.81, 0.05)
        values.append(float(max(candidates, key=lambda t: (f1_score(y[:, j], p[:, j] >= t, zero_division=0), -abs(t - .5)))))
    return values


def evaluate_a(model, loader, device, thresholds=None):
    y, p, records = [], [], []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            request, needs = model(**{k: batch[k].to(device) for k in ['input_ids', 'attention_mask']})
            scores = torch.cat([request.sigmoid()[:, None], needs.sigmoid()], dim=1).cpu().numpy()
            y.extend(batch['targets'].numpy())
            p.extend(scores)
            records.extend({'id': item['row']['id'], 'scores': score.tolist()} for item, score in zip(batch['items'], scores))
    y, p = np.array(y).astype(int), np.array(p)
    thresholds = thresholds or choose_thresholds(y, p)
    predicted = p >= np.array(thresholds)
    gated = predicted[:, 1:] & predicted[:, 0, None]
    metrics = {
        'request': classification_report(y[:, 0], predicted[:, 0], labels=[0, 1], output_dict=True, zero_division=0),
        'needs': classification_report(y[:, 1:], predicted[:, 1:], target_names=NEEDS, output_dict=True, zero_division=0),
        'needs_macro_f1': f1_score(y[:, 1:], predicted[:, 1:], average='macro', zero_division=0),
        'needs_micro_f1': f1_score(y[:, 1:], predicted[:, 1:], average='micro', zero_division=0),
        'gated_needs_macro_f1': f1_score(y[:, 1:], gated, average='macro', zero_division=0),
        'thresholds': thresholds,
    }
    metrics['selection_score'] = (metrics['needs_macro_f1'] + f1_score(y[:, 0], predicted[:, 0], zero_division=0)) / 2
    return metrics, records


def evaluate_b(model, loader, device):
    counts = {kind: [0, 0, 0] for kind in ENTITIES}
    records = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            logits = model(**{k: batch[k].to(device) for k in ['input_ids', 'attention_mask']}).logits.cpu()
            for item, output in zip(batch['items'], logits):
                row = item['row']
                tags = [BIO[output[i].argmax().item()] for i in item['first']]
                spans = decode_spans(row['text'], item['words'], tags)
                predicted = {(e['start'], e['end'], e['label']) for e in spans}
                gold = {(e['start'], e['end'], e['label']) for e in row['entities']}
                for kind in ENTITIES:
                    pred_k, gold_k = {e for e in predicted if e[2] == kind}, {e for e in gold if e[2] == kind}
                    for i, value in enumerate([len(pred_k & gold_k), len(pred_k), len(gold_k)]):
                        counts[kind][i] += value
                records.append({'id': row['id'], 'entities': spans, 'gold': row['entities']})
    def scores(values):
        tp, pred, gold = values
        return {'precision': tp / pred if pred else 0, 'recall': tp / gold if gold else 0,
                'f1': 2 * tp / (pred + gold) if pred + gold else 0, 'gold_count': gold}
    metrics = {kind: scores(values) for kind, values in counts.items()}
    metrics['micro'] = scores([sum(v[i] for v in counts.values()) for i in range(3)])
    metrics['selection_score'] = metrics['micro']['f1']
    return metrics, records


## 6. Quy trình train, đánh giá, dự đoán và xuất model

Đây là code thực thi, được đặt ngay trong notebook để có thể đọc và trình bày.
Các bước bên dưới gọi từng thao tác riêng. `continue` nạp cả encoder và đầu tác vụ từ model cha, tạo optimizer mới.
Chỉ Validation được dùng để chọn checkpoint. Trong chế độ học tiếp, model cha vẫn là ứng viên nếu train thêm không cải thiện Validation.

In [ ]:
"""Visible training steps embedded into Colab, also importable for offline tests."""
import gc
import hashlib
import json
import math
import platform
import random
import shutil
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForTokenClassification
from tqdm.auto import tqdm


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


class NotebookRun:
    def __init__(self, task, splits, workbook_path, output, *, parent=None,
                 epochs=5, batch_size=8, lr=2e-5, seed=42, evaluation_domain='synthetic'):
        if task not in {'a', 'b'} or type(epochs) is not int or epochs < 1:
            raise ValueError('Task/epochs không hợp lệ')
        if type(batch_size) is not int or batch_size < 1 or not math.isfinite(lr) or lr <= 0:
            raise ValueError('Batch size và learning rate phải là số dương hữu hạn')
        self.task, self.splits, self.workbook_path = task, splits, Path(workbook_path)
        self.output, self.parent_path = Path(output), Path(parent) if parent else None
        if self.output.exists():
            raise ValueError(f'Run đã tồn tại: {self.output}. Đổi RUN_NAME để giữ kết quả cũ.')
        self.epochs, self.batch_size, self.lr, self.seed = epochs, batch_size, lr, seed
        self.domain, self.backbone = evaluation_domain, 'vinai/phobert-base-v2'
        self.checkpoint = self.output / f'model_{task}'
        self.evaluate = evaluate_a if task == 'a' else evaluate_b
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.ready = self.baseline_done = self.training_done = self.evaluation_done = False

    def prepare(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)
        parent, lineage = (read_parent(self.parent_path, self.task) if self.parent_path else (None, None))
        if parent:
            check_excel_parent(parent, self.splits, self.task)
        self.parent = parent
        self.tokenizer = AutoTokenizer.from_pretrained(str(self.parent_path or self.backbone),
            use_fast=False, local_files_only=bool(parent))
        prepared = {}
        for split, rows in self.splits.items():
            prepared[split] = []
            for index, row in enumerate(tqdm(rows, desc=f'Kiểm tra {split}'), 2):
                try:
                    item = encode(row['text'], self.tokenizer, row['entities'] if self.task == 'b' else None)
                except ValueError as exc:
                    raise ValueError(f'{split}, dòng Excel {index}, ID={row["id"]}: {exc}') from exc
                item['row'] = row
                prepared[split].append(item)
        self.loaders = {name: DataLoader(rows, batch_size=self.batch_size, shuffle=name == 'train',
            collate_fn=lambda items: collate(items, self.tokenizer.pad_token_id)) for name, rows in prepared.items()}
        if parent:
            self.model = load_parent_model(self.parent_path, self.task)
        elif self.task == 'a':
            self.model = RequestNeedsModel(AutoModel.from_pretrained(self.backbone))
        else:
            self.model = AutoModelForTokenClassification.from_pretrained(self.backbone,
                num_labels=len(BIO), id2label=dict(enumerate(BIO)), label2id={v:i for i,v in enumerate(BIO)})
        self.model.to(self.device)
        self.metadata = {'schema_version': 1, 'run_id': self.output.name, 'task': self.task,
            'created_at': datetime.now(timezone.utc).isoformat(), 'backbone': parent['backbone'] if parent else self.backbone,
            'preprocessor': PREPROCESSOR, 'max_length': MAX_LENGTH, 'needs': NEEDS, 'bio_labels': BIO,
            'demo': False, 'evaluation_domain': self.domain,
            'initialization': 'continued_fine_tuning' if parent else 'pretrained_backbone_random_task_heads',
            'parent_checkpoint': lineage, 'optimizer_resumed': False,
            'hyperparameters': {'epochs': self.epochs, 'batch_size': self.batch_size, 'lr': self.lr, 'seed': self.seed},
            'versions': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__},
            'backbone_revision': getattr(self.model.encoder.config if self.task == 'a' else self.model.config, '_commit_hash', None),
            'excel_split_sha256': {s: semantic_hash(rows, self.task) for s, rows in self.splits.items()},
            'data_counts': {s: len(rows) for s, rows in self.splits.items()},
            'data_origins': {s: {origin: sum(r['origin'] == origin for r in rows) for origin in ['real', 'synthetic', 'translated']} for s, rows in self.splits.items()}}
        self.output.mkdir(parents=True, exist_ok=False)
        shutil.copy2(self.workbook_path, self.output / self.workbook_path.name)
        self.ready = True
        self._status('prepared')
        print(f'Sẵn sàng: {self.device}, {self.metadata["data_counts"]}\nLưu tại: {self.output}')

    def _status(self, phase):
        save_json(self.output / 'status.json', {'phase': phase, 'run': self.output.name})

    def _threshold_kwargs(self, metadata):
        if self.task == 'a' and 'thresholds' in metadata:
            return {'thresholds': [metadata['thresholds']['request']] + metadata['thresholds']['needs']}
        return {}

    def measure_before(self):
        if not self.ready:
            raise RuntimeError('Chạy bước chuẩn bị model trước.')
        if self.baseline_done:
            print('Đã đo baseline, giữ nguyên kết quả trước train.')
            return self.before
        parent_thresholds = self._threshold_kwargs(self.parent or {})
        self.before_validation, _ = self.evaluate(self.model, self.loaders['validation'], self.device, **parent_thresholds)
        kwargs = {'thresholds': self.before_validation['thresholds']} if self.task == 'a' else {}
        self.before, predictions = self.evaluate(self.model, self.loaders['test'], self.device, **kwargs)
        self.before['evaluation_domain'] = self.domain
        self.before['baseline_type'] = 'parent_model' if self.parent else 'pretrained_encoder_random_task_heads'
        save_json(self.output / 'before_metrics.json', self.before)
        write_jsonl(self.output / 'before_predictions.jsonl', predictions)
        self.baseline_done = True
        self._status('baseline_measured')
        return self.before

    def _save_best(self, metrics, epoch):
        self.metadata['best_epoch'] = epoch
        if self.task == 'a':
            t = metrics['thresholds']
            self.metadata['thresholds'] = {'request': t[0], 'needs': t[1:]}
            self.model.save_bundle(self.checkpoint, self.tokenizer, self.metadata)
        else:
            self.model.save_pretrained(self.checkpoint, safe_serialization=True)
            self.tokenizer.save_pretrained(self.checkpoint)
            save_json(self.checkpoint / 'metadata.json', self.metadata)

    def train(self):
        if not self.baseline_done:
            raise RuntimeError('Chạy bước đánh giá trước fine-tune trước khi train.')
        if self.training_done or getattr(self, 'training_started', False):
            raise RuntimeError('Run này đã train hoặc từng bị gián đoạn. Tạo RUN_NAME mới; dùng continue từ checkpoint nếu có.')
        self.training_started = True
        self._status('training')
        best, self.history = -1.0, []
        if self.parent:
            best = self.before_validation['selection_score']
            self._save_best(self.before_validation, 0)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=.01)
        # A fresh optimizer is intentional for a new data version (continued fine-tuning).
        try:
            for epoch in range(1, self.epochs + 1):
                self.model.train()
                losses = []
                progress = tqdm(self.loaders['train'], desc=f'Epoch {epoch}/{self.epochs}')
                for batch in progress:
                    optimizer.zero_grad(set_to_none=True)
                    inputs = {k: batch[k].to(self.device) for k in ['input_ids', 'attention_mask']}
                    if self.task == 'a':
                        request, needs = self.model(**inputs)
                        targets = batch['targets'].to(self.device)
                        loss = torch.nn.functional.binary_cross_entropy_with_logits(request, targets[:, 0])
                        loss = loss + torch.nn.functional.binary_cross_entropy_with_logits(needs, targets[:, 1:])
                    else:
                        loss = self.model(**inputs, labels=batch['labels'].to(self.device)).loss
                    if not torch.isfinite(loss):
                        raise RuntimeError('Loss không hữu hạn. Kiểm tra dữ liệu và learning rate.')
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    losses.append(loss.item())
                    progress.set_postfix(loss=f'{loss.item():.4f}')
                metrics, _ = self.evaluate(self.model, self.loaders['validation'], self.device)
                self.history.append({'epoch': epoch, 'loss': float(np.mean(losses)), 'validation': metrics})
                if metrics['selection_score'] > best:
                    best = metrics['selection_score']
                    self._save_best(metrics, epoch)
                save_json(self.output / 'history.json', self.history)
                print(f'Epoch {epoch}: loss={np.mean(losses):.4f}, validation={metrics["selection_score"]:.4f}, best={best:.4f}')
            self.training_done = True
            self._status('trained')
        except BaseException:
            self._status('interrupted')
            raise
        finally:
            del optimizer
            gc.collect()

    def finish(self):
        if not self.training_done:
            raise RuntimeError('Train chưa hoàn thành. Xem lỗi tại ô Train; chưa có báo cáo test cuối cùng.')
        if self.evaluation_done:
            return self.comparison
        del self.model
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
        self.model = load_parent_model(self.checkpoint, self.task).to(self.device)
        self.saved = json.loads((self.checkpoint / 'metadata.json').read_text(encoding='utf-8'))
        after, predictions = self.evaluate(self.model, self.loaders['test'], self.device, **self._threshold_kwargs(self.saved))
        after.update(evaluation_domain=self.domain, demo_not_research_result=False,
                     synthetic_only=self.domain == 'synthetic', best_epoch=self.saved['best_epoch'])
        save_json(self.output / 'test_metrics.json', after)
        write_jsonl(self.output / 'test_predictions.jsonl', predictions)
        def compact(report):
            if self.task == 'a':
                return {'Request F1': report['request']['1']['f1-score'], 'Needs macro F1': report['needs_macro_f1'],
                        'Needs micro F1': report['needs_micro_f1'], 'Needs F1 sau lọc yêu cầu': report['gated_needs_macro_f1']}
            return {'Span micro F1': report['micro']['f1'], **{label + ' F1': report[label]['f1'] for label in ENTITIES}}
        before_values, after_values = compact(self.before), compact(after)
        self.comparison = [{'Chỉ số': k, 'Trước': before_values[k], 'Sau': after_values[k],
                            'Chênh lệch': after_values[k] - before_values[k]} for k in before_values]
        save_json(self.output / 'comparison.json', {'baseline_type': self.before['baseline_type'],
            'evaluation_domain': self.domain, 'best_epoch': self.saved['best_epoch'], 'metrics': self.comparison})
        self.evaluation_done = True
        self._status('evaluated')
        return self.comparison

    def predict(self, text):
        if not self.evaluation_done:
            raise RuntimeError('Hoàn thành đánh giá model tốt nhất trước khi thử câu mới.')
        text = unicodedata.normalize('NFC', text)
        item = encode(text, self.tokenizer)
        inputs = {k: torch.tensor([item[k]], device=self.device) for k in ['input_ids', 'attention_mask']}
        self.model.eval()
        with torch.inference_mode():
            if self.task == 'a':
                req, needs = self.model(**inputs)
                score, scores = req.sigmoid().item(), needs.sigmoid()[0].tolist()
                thresholds = self.saved['thresholds']
                request = score >= thresholds['request']
                return {'text': text, 'is_request': request, 'request_score': score,
                    'needs': [n for n, p, t in zip(NEEDS, scores, thresholds['needs']) if request and p >= t],
                    'need_scores': dict(zip(NEEDS, scores))}
            logits = self.model(**inputs).logits[0]
            tags = [BIO[logits[i].argmax().item()] for i in item['first']]
            return {'text': text, 'entities': decode_spans(text, item['words'], tags)}

    def export(self):
        if not self.evaluation_done:
            raise RuntimeError('Chạy đánh giá sau train trước khi xuất model.')
        # The archive contains model_a/ OR model_b/, matching SOSPredictor's root layout.
        archive = shutil.make_archive(str(self.output / f'model_{self.task}'), 'zip',
                                      root_dir=self.output, base_dir=f'model_{self.task}')
        digest = hashlib.sha256()
        with open(archive, 'rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        save_json(self.output / 'export.json', {'archive': str(archive), 'sha256': digest.hexdigest()})
        print(f'Model đã lưu trên Drive: {self.checkpoint}\nZIP để tích hợp: {archive}')
        return archive


In [ ]:
session = NotebookRun(TASK, SPLITS, EXCEL, RUN, parent=PARENT,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED,
    evaluation_domain=EVALUATION_DOMAIN)
session.prepare()


## 7. Đánh giá trước fine-tune

`new`: encoder PhoBERT gốc + đầu tác vụ ngẫu nhiên. `continue`: model của lần trước.
Lưu `before_metrics.json`. Không dùng kết quả Test này để chọn learning rate, epoch hoặc ngưỡng.

In [ ]:
before = session.measure_before()
print(json.dumps(before, ensure_ascii=False, indent=2))


## 8. Fine-tune

Đây là ô thực sự huấn luyện. Thanh tiến trình hiển thị từng batch; sau mỗi epoch có loss và F1 Validation.
Checkpoint tốt nhất được lưu vào Drive sau mỗi lần cải thiện.
Nếu lỗi/mất kết nối: không chạy ô đọc báo cáo để bỏ qua lỗi. Xem hướng dẫn xử lý ở cuối notebook.

In [ ]:
session.train()


## 9. Đánh giá sau fine-tune và so sánh

Nạp lại checkpoint tốt nhất rồi đo Test với ngưỡng đã chọn trên Validation.
Bảng dùng thang 0–1. Chênh lệch có thể âm; notebook không tạo hoặc làm đẹp điểm số.
File `test_metrics.json` có chi tiết từng nhãn, `test_predictions.jsonl` có dự đoán từng mẫu.

In [ ]:
comparison = pd.DataFrame(session.finish()).set_index('Chỉ số')
display(comparison.style.format('{:.4f}'))
print('Epoch được chọn:', session.saved['best_epoch'])
print('Phạm vi đánh giá:', EVALUATION_DOMAIN)
import matplotlib.pyplot as plt
ax = comparison[['Trước', 'Sau']].plot.barh(figsize=(9, 4), xlim=(0, 1))
ax.set_xlabel('F1 (0–1)')
ax.set_title('Đánh giá ' + EVALUATION_DOMAIN + ' — trước và sau fine-tune')
plt.tight_layout()
plt.savefig(RUN / 'comparison.png', dpi=160, bbox_inches='tight')
plt.show()
history = pd.DataFrame([{'Epoch': h['epoch'], 'Loss': h['loss'], 'Validation F1': h['validation']['selection_score']} for h in session.history])
display(history)


## 10. Thử câu mới

In [ ]:
TEXT = 'Nhờ cứu 5 người tại xã An Bình, có 2 trẻ em, đang cần nước uống.' #@param {type:"string"}
print(json.dumps(session.predict(TEXT), ensure_ascii=False, indent=2))


## 11. Xuất model để tích hợp

Trọng số đã nằm trên Drive trong `runs/<tên run>/model_a`.
Ô này tạo **ZIP model đầu ra** để tải về, không phải ZIP code đầu vào.
Sau khi có cả A và B, giải nén hai ZIP vào cùng thư mục `sos_two_model_v1`, bên trong có `model_a/` và `model_b/`.
Cấu hình backend: `SOS_TWO_MODEL_DIR` trỏ tới thư mục đó; `SOS_TWO_MODEL_DEVICE=cpu` hoặc `cuda`.
API hiện có: `POST /ai/analyze-sos/`, JSON `{"text": "Nhờ cứu gia đình tôi..."}`, cần tài khoản có quyền admin.
API test trả gợi ý và không tạo SOS. Crawler Facebook dùng hai model; admin xác nhận thông tin trước khi tạo SOS. Trên web admin, mở AI Tìm tin → Test model A / B để thử từng model hoặc cả hai. Xem README để kiểm tra tích hợp bằng Python.

In [ ]:
DOWNLOAD_MODEL = False #@param {type:"boolean"}
archive = session.export()
if DOWNLOAD_MODEL:
    files.download(archive)
print('Đường dẫn dùng cho PARENT_MODEL lần sau:', session.checkpoint)


## 12. Thêm dữ liệu và học tiếp bằng chính notebook này

1. Mở `data_model_a.xlsx`, thêm mẫu vào **Train**. Giữ các mẫu cũ để hạn chế quên kiến thức.
2. Không đổi Validation/Test. Mẫu cùng bài gốc/sự kiện/mẫu câu dùng cùng `group_id`; không đưa mẫu test vào train.
3. Mở lại notebook. Ở bước 2, chọn `TRAIN_MODE=continue`, dán đường dẫn model_a đã xuất vào `PARENT_MODEL`.
4. Đặt `EPOCHS=3`, `LEARNING_RATE=0.000005` làm điểm bắt đầu. Để `RUN_NAME` trống hoặc chọn tên mới.
5. Đặt `UPLOAD_EXCEL=True`, upload Excel mới và chạy lần lượt đến bước 11.

Model dùng lại trọng số cũ, fine-tune trên dữ liệu cũ + mới; **không khởi tạo lại từ PhoBERT gốc**.
Optimizer được tạo mới, nên đây không phải tiếp tục đúng batch đã bị ngắt. Không thêm loại nhãn mới bằng cách thêm cột.
Model từ notebook ZIP cũ không có dấu vân tay đánh giá Excel nên không được nhận làm model cha trong quy trình này.

## Khi có lỗi

- **Ô số báo không hợp lệ:** chỉ nhập số, ví dụ `5`, `8`, `0.00002`.
- **Sai Transformers sau cài:** restart session, chạy từ bước 1. Không sửa assert để bỏ qua.
- **Excel sai:** sửa đúng dòng được báo, upload lại; chưa cần train lại nếu lỗi ở kiểm tra dữ liệu.
- **CUDA out of memory:** giảm `BATCH_SIZE` xuống 4 hoặc 2, restart session, dùng tên run mới.
- **Train gián đoạn:** nếu Drive đã có `model_a/metadata.json` cùng trọng số đầy đủ, tạo run mới dùng `continue` từ thư mục đó.
  Nếu chưa lưu được checkpoint hoàn chỉnh, chạy `new` lại. Không xóa run cũ để ép chạy tiếp.
- **Chưa có test_metrics:** chỉ có sau bước 9 thành công. Sửa lỗi train trước; notebook sẽ báo trạng thái thay vì đọc file chưa có.

Tài liệu: [PhoBERT](https://github.com/VinAIResearch/PhoBERT),
[Transformers 4.57.6](https://pypi.org/project/transformers/4.57.6/),
[Tokenizers 0.22.2](https://pypi.org/project/tokenizers/0.22.2/).